# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [2]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [3]:
baseline_rows = len(df)
df['revenue']=df['qty']*df['price']
total_revenue=df['revenue'].sum()
total_units = df['qty'].sum()
print(total_revenue) #this is the total revenue from multiplying the quantity by price.
print(total_units) #this is the number of total units (sum of quantities)

8520.0
783


There is a total revenue of $8520 from the quantity x price and there are 783 units.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [12]:
revenue_by_category=df.groupby('category')['revenue'].sum()
revenue_by_category =revenue_by_category.sort_values(ascending=False)
share_of_total=revenue_by_category/total_revenue *100

table = pd.DataFrame({
    'revenue': revenue_by_category,
    'share_of_total': share_of_total
})
print(table)

          revenue  share_of_total
category                         
Food       4293.0       50.387324
Merch      1771.5       20.792254
Drink      1554.0       18.239437
RainGear    901.5       10.580986


The table above displays the revenues by category (food, merch, drink, and raingear), and share of total is displayed as a percentage of the total revenue on the right side of the table. The revenues are sorted from highest to lowest.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [13]:
average_revenue = df.groupby('vendor_id')['revenue'].mean()
order_count = df.groupby('vendor_id')['revenue'].count()

table = pd.DataFrame({
    'average_revenue': average_revenue,
    'order_count': order_count
})
print(table)

           average_revenue  order_count
vendor_id                              
V-01             22.595745           94
V-05             20.580645           93
V-10             20.314286          105
V-18             21.750000          108


In the table above, the different vendors are listed on the left side, the average revenue is the middle column taken by the .mean() function, and the order_count is taken by the .count() function on the rightmost column.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [15]:
revenue_from_merch = df[df['category'] == 'Merch']['revenue'].sum()
share_of_revenue_from_merch = revenue_from_merch / total_revenue * 100
print(share_of_revenue_from_merch.round(1))

20.8


This percentage - 20.8 - represents the percent of revenue that is only generated from merch.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [22]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
clean_vendors = df.merge(vendor_names, on='vendor_id',how='left', validate = 'many_to_one')

missing = clean_vendors[clean_vendors['vendor_name'].isna()]
print(f'unmatched orders: {len(missing)}')
print(missing['vendor_id'])
print(f'revenue at stake: ${missing["revenue"].sum():.2f}')
clean_vendors['vendor_name'] = clean_vendors['vendor_name'].fillna('Unknown vendor')

print(len(df))
print(len(clean_vendors))
print(df['revenue'].sum())
print(clean_vendors['revenue'].sum())



unmatched orders: 108
1      V-18
2      V-18
4      V-18
5      V-18
6      V-18
       ... 
384    V-18
386    V-18
389    V-18
395    V-18
398    V-18
Name: vendor_id, Length: 108, dtype: object
revenue at stake: $2349.00
400
400
8520.0
8520.0


**The unmatched vendor, and what I did about it:** I labeled the vendor "unknown vendor and then I kept it in a zone where it was unmatched. The unmatched vendor was V-18, which appeared 108 times and there could be a lot of revenue at stake here.

**bold text**### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [23]:
pivot = pd.pivot_table(
    clean_vendors,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True
)

print(pivot)

category          Drink    Food   Merch  RainGear     All
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
Unknown vendor    582.0  1018.5   508.5     240.0  2349.0
All              1554.0  4293.0  1771.5     901.5  8520.0


I added in row and column totals at the ends. This pivot table shows all drinks, food, merch, and raingear.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [28]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(revenue_by_category.sum() - df['revenue'].sum()) < 0.01
assert len(clean_vendors) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

I would tell the vendors to focus more on categories that made the most revenue so that for future games, they can maximize their revenue, as food made up $4293, making up 50% of the total money.I would also tell the vendors to pay attention to the second category, as this made up around more than 20 percent of sales.

My least trustworthy answer is the vendor revenue comparison because one vendor was not included in the vendor lookup. There were 108 orders from the unmatched vendor, which means those orders had to be labeled as "unknown vendor. The revenue from those orders is still included in the overall totals. This makes conclusions about individual vendor performance less reliable.